# Streamlit-приложение для прогноза свойств композитов — запуск в Google Colab

ВКР по курсу «Data Science Pro», МГТУ им. Н. Э. Баумана, 2025.

**Последовательность запуска:**
1. `Runtime → Run all`, либо выполнять ячейки по порядку.
2. Когда потребуется — загрузить в директорию `data/` два файла: `X_bp.xlsx` и `X_nup.xlsx`.
3. Дождаться обучения моделей (≈1–2 минуты).
4. В последней ячейке получить публичный HTTPS-URL и открыть его в новой вкладке.

**Зачем нужен тоннель.** Streamlit слушает `localhost:8501` внутри виртуальной машины Colab. Этот адрес недоступен вашему браузеру напрямую, поэтому порт 8501 пробрасывается наружу через сервис ngrok (либо localtunnel).


## 1. Установка зависимостей


In [ ]:
!pip install -q streamlit==1.39 torch scikit-learn xgboost joblib openpyxl pyngrok

import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('✓ Зависимости установлены, директории data/ и models/ созданы')


## 2. Загрузка датасета

Два способа на выбор — выполните **только один** из двух следующих блоков.

### Способ A. Загрузка с диска компьютера через диалог
В диалоге выберите сразу оба файла: `X_bp.xlsx` и `X_nup.xlsx`.


In [ ]:
from google.colab import files
import shutil, os

uploaded = files.upload()
for name in uploaded:
    if name.endswith('.xlsx'):
        shutil.move(name, f'data/{name}')

print('Файлы в data/:', os.listdir('data'))


### Способ B. Подключение Google Drive (если датасет лежит там)
При первом запуске Colab попросит авторизацию. Путь в `shutil.copy` — отредактируйте под себя.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/vkr/X_bp.xlsx', 'data/')
# shutil.copy('/content/drive/MyDrive/vkr/X_nup.xlsx', 'data/')
# import os; print('Файлы в data/:', os.listdir('data'))


## 3. Сохранение `train_models.py` на диск ВМ Colab


In [ ]:
%%writefile train_models.py
"""
train_models.py — обучение и сохранение итоговых моделей ВКР.

Модели сохраняются в директорию models/:
    ridge_modulus.joblib        — Ridge-регрессия для модуля упругости при растяжении
    rf_strength.joblib          — Random Forest для прочности при растяжении
    scaler_23.joblib            — StandardScaler для входов моделей раздела 2.3
    mlp_ratio.pt                — state_dict нейронной сети (PyTorch)
    scaler_mlp_X.joblib         — StandardScaler для входов нейронной сети (раздел 2.4)
    scaler_mlp_y.joblib         — StandardScaler для выхода нейронной сети
    metadata.json               — имена признаков, метрики, версия
"""
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

TARGET_MODULUS = "Модуль упругости при растяжении, ГПа"
TARGET_STRENGTH = "Прочность при растяжении, МПа"
TARGET_RATIO = "Соотношение матрица-наполнитель"

# Порядок признаков фиксируется явно, чтобы приложение и обучение были синхронны
FEATURES_23 = [
    "Соотношение матрица-наполнитель",
    "Плотность, кг/м3",
    "модуль упругости, ГПа",
    "Количество отвердителя, м.%",
    "Содержание эпоксидных групп,%_2",
    "Температура вспышки, С_2",
    "Поверхностная плотность, г/м2",
    "Потребление смолы, г/м2",
    "Угол нашивки, град",
    "Шаг нашивки",
    "Плотность нашивки",
]
FEATURES_24 = [f for f in FEATURES_23 if f != TARGET_RATIO]


class MLPRatio(nn.Module):
    """Архитектура MLP-2.4."""

    def __init__(self, n_in: int = 10) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.BatchNorm1d(32), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def load_dataset() -> pd.DataFrame:
    """Загружает и объединяет два Excel-файла датасета по индексу (INNER JOIN)."""
    df_bp = pd.read_excel(DATA_DIR / "X_bp.xlsx", index_col=0)
    df_nup = pd.read_excel(DATA_DIR / "X_nup.xlsx", index_col=0)
    df = df_bp.join(df_nup, how="inner")
    if df.isna().any().any() or df.duplicated().any():
        raise ValueError("Датасет содержит пропуски или дубликаты")
    return df


def train_regression_models(df: pd.DataFrame) -> dict[str, dict]:
    """Обучает Ridge (для модуля) и RandomForest (для прочности) и возвращает метрики."""
    X = df[FEATURES_23].values
    y_mod = df[TARGET_MODULUS].values
    y_str = df[TARGET_STRENGTH].values

    scaler = StandardScaler().fit(X)
    X_s = scaler.transform(X)

    cv = KFold(n_splits=10, shuffle=True, random_state=SEED)

    # Ridge для модуля упругости (минимум переобучения, ближе всего к baseline)
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_mod, test_size=0.3, random_state=SEED)
    ridge_mod = Ridge(alpha=10.0, random_state=SEED).fit(Xtr, ytr)
    metrics_mod = _metrics(ridge_mod, X_s, y_mod, Xtr, ytr, Xte, yte, cv)

    # Random Forest для прочности (единственная модель, превосходящая baseline по R²(CV))
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_str, test_size=0.3, random_state=SEED)
    rf_str = RandomForestRegressor(
        n_estimators=200, max_depth=5, min_samples_leaf=5,
        random_state=SEED, n_jobs=-1,
    ).fit(Xtr, ytr)
    metrics_str = _metrics(rf_str, X_s, y_str, Xtr, ytr, Xte, yte, cv)

    joblib.dump(ridge_mod, MODELS_DIR / "ridge_modulus.joblib")
    joblib.dump(rf_str, MODELS_DIR / "rf_strength.joblib")
    joblib.dump(scaler, MODELS_DIR / "scaler_23.joblib")

    return {"modulus": metrics_mod, "strength": metrics_str}


def _metrics(model, X_full, y_full, Xtr, ytr, Xte, yte, cv) -> dict:
    ptr, pte = model.predict(Xtr), model.predict(Xte)
    cv_r2 = cross_val_score(model, X_full, y_full, cv=cv, scoring="r2", n_jobs=-1).mean()
    return {
        "MAE_test": round(mean_absolute_error(yte, pte), 4),
        "RMSE_test": round(float(np.sqrt(mean_squared_error(yte, pte))), 4),
        "R2_test": round(r2_score(yte, pte), 4),
        "R2_CV": round(float(cv_r2), 4),
        "R2_train": round(r2_score(ytr, ptr), 4),
    }


def train_mlp(df: pd.DataFrame) -> dict:
    """Обучает MLP для рекомендации соотношения матрица-наполнитель."""
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    X = df[FEATURES_24].values.astype(np.float32)
    y = df[TARGET_RATIO].values.astype(np.float32)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED)
    Xtr, Xval, ytr, yval = train_test_split(Xtr, ytr, test_size=0.15, random_state=SEED)

    x_scaler = StandardScaler().fit(Xtr)
    y_scaler = StandardScaler().fit(ytr.reshape(-1, 1))
    Xtr_s = x_scaler.transform(Xtr).astype(np.float32)
    Xval_s = x_scaler.transform(Xval).astype(np.float32)
    Xte_s = x_scaler.transform(Xte).astype(np.float32)
    ytr_s = y_scaler.transform(ytr.reshape(-1, 1)).astype(np.float32).ravel()
    yval_s = y_scaler.transform(yval.reshape(-1, 1)).astype(np.float32).ravel()

    model = MLPRatio(n_in=len(FEATURES_24))
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr_s), torch.tensor(ytr_s).unsqueeze(1)),
        batch_size=32, shuffle=True,
    )
    Xval_t, yval_t = torch.tensor(Xval_s), torch.tensor(yval_s).unsqueeze(1)

    best_val, patience_cnt, best_state = float("inf"), 0, None
    for _ in range(300):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            nn.functional.mse_loss(model(xb), yb).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = nn.functional.mse_loss(model(Xval_t), yval_t).item()
        scheduler.step(val_loss)
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= 30:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_s = model(torch.tensor(Xte_s)).numpy().ravel()
    pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()

    torch.save(model.state_dict(), MODELS_DIR / "mlp_ratio.pt")
    joblib.dump(x_scaler, MODELS_DIR / "scaler_mlp_X.joblib")
    joblib.dump(y_scaler, MODELS_DIR / "scaler_mlp_y.joblib")

    return {
        "MAE_test": round(mean_absolute_error(yte, pred), 4),
        "RMSE_test": round(float(np.sqrt(mean_squared_error(yte, pred))), 4),
        "R2_test": round(r2_score(yte, pred), 4),
    }


def main() -> None:
    print("Загрузка датасета …")
    df = load_dataset()
    print(f"  размер: {df.shape[0]} строк × {df.shape[1]} столбцов\n")

    print("Обучение регрессионных моделей (Ridge, Random Forest) …")
    reg_metrics = train_regression_models(df)
    for target, m in reg_metrics.items():
        print(f"  {target}: {m}")

    print("\nОбучение нейронной сети MLP для соотношения матрица-наполнитель …")
    mlp_metrics = train_mlp(df)
    print(f"  ratio: {mlp_metrics}")

    metadata = {
        "version": "1.0",
        "features_23": FEATURES_23,
        "features_24": FEATURES_24,
        "target_modulus": TARGET_MODULUS,
        "target_strength": TARGET_STRENGTH,
        "target_ratio": TARGET_RATIO,
        "metrics": {**reg_metrics, "ratio": mlp_metrics},
        "seed": SEED,
    }
    (MODELS_DIR / "metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    print(f"\n✓ Все модели сохранены в {MODELS_DIR.resolve()}")


if __name__ == "__main__":
    main()


## 4. Сохранение `app.py` на диск ВМ Colab


In [ ]:
%%writefile app.py
"""Streamlit-приложение для прогнозирования свойств композиционных материалов.

Приложение загружает предварительно обученные модели из директории models/
и предоставляет два независимых режима работы:
    1) прогноз модуля упругости при растяжении и прочности при растяжении;
    2) рекомендацию соотношения матрица-наполнитель.

Запуск:
    streamlit run app.py
"""
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st
import torch
import torch.nn as nn

MODELS_DIR = Path("models")


class MLPRatio(nn.Module):
    """Архитектура MLP-2.4 должна соответствовать train_models.py."""

    def __init__(self, n_in: int = 10) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.BatchNorm1d(32), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


@st.cache_resource
def load_artifacts():
    """Загружает все модели один раз и кеширует до перезапуска процесса."""
    if not MODELS_DIR.exists():
        st.error(
            "Директория `models/` не найдена. "
            "Запустите `python train_models.py` для создания артефактов."
        )
        st.stop()

    metadata = json.loads((MODELS_DIR / "metadata.json").read_text(encoding="utf-8"))
    ridge_mod = joblib.load(MODELS_DIR / "ridge_modulus.joblib")
    rf_str = joblib.load(MODELS_DIR / "rf_strength.joblib")
    scaler_23 = joblib.load(MODELS_DIR / "scaler_23.joblib")

    mlp = MLPRatio(n_in=len(metadata["features_24"]))
    mlp.load_state_dict(torch.load(MODELS_DIR / "mlp_ratio.pt", map_location="cpu"))
    mlp.eval()
    scaler_mlp_X = joblib.load(MODELS_DIR / "scaler_mlp_X.joblib")
    scaler_mlp_y = joblib.load(MODELS_DIR / "scaler_mlp_y.joblib")

    return {
        "metadata": metadata,
        "ridge_mod": ridge_mod,
        "rf_str": rf_str,
        "scaler_23": scaler_23,
        "mlp": mlp,
        "scaler_mlp_X": scaler_mlp_X,
        "scaler_mlp_y": scaler_mlp_y,
    }


# Реалистичные значения по умолчанию — средние по обучающей выборке
DEFAULTS = {
    "Соотношение матрица-наполнитель": 2.93,
    "Плотность, кг/м3": 1975.0,
    "модуль упругости, ГПа": 740.0,
    "Количество отвердителя, м.%": 110.6,
    "Содержание эпоксидных групп,%_2": 22.2,
    "Температура вспышки, С_2": 285.9,
    "Поверхностная плотность, г/м2": 482.7,
    "Потребление смолы, г/м2": 218.4,
    "Угол нашивки, град": 0,
    "Шаг нашивки": 6.9,
    "Плотность нашивки": 57.2,
}

INPUT_SPECS = {
    "Соотношение матрица-наполнитель": dict(min=0.0, max=10.0, step=0.1, fmt="%.2f"),
    "Плотность, кг/м3": dict(min=1500.0, max=2500.0, step=5.0, fmt="%.1f"),
    "модуль упругости, ГПа": dict(min=0.0, max=2000.0, step=5.0, fmt="%.1f"),
    "Количество отвердителя, м.%": dict(min=0.0, max=250.0, step=1.0, fmt="%.1f"),
    "Содержание эпоксидных групп,%_2": dict(min=10.0, max=40.0, step=0.1, fmt="%.2f"),
    "Температура вспышки, С_2": dict(min=50.0, max=500.0, step=1.0, fmt="%.1f"),
    "Поверхностная плотность, г/м2": dict(min=0.0, max=1500.0, step=5.0, fmt="%.1f"),
    "Потребление смолы, г/м2": dict(min=0.0, max=500.0, step=1.0, fmt="%.1f"),
    "Угол нашивки, град": dict(options=[0, 90]),
    "Шаг нашивки": dict(min=0.0, max=20.0, step=0.1, fmt="%.1f"),
    "Плотность нашивки": dict(min=0.0, max=150.0, step=1.0, fmt="%.1f"),
}


def render_input(feature: str, key_prefix: str):
    """Отображает поле ввода, соответствующее типу признака."""
    spec = INPUT_SPECS[feature]
    key = f"{key_prefix}::{feature}"
    if "options" in spec:
        return st.selectbox(feature, options=spec["options"], key=key)
    return st.number_input(
        feature,
        min_value=spec["min"], max_value=spec["max"],
        value=float(DEFAULTS[feature]), step=spec["step"], format=spec["fmt"],
        key=key,
    )


def show_model_disclaimer(metrics: dict, context: str) -> None:
    """Отображает честное предупреждение о качестве модели."""
    r2 = metrics.get("R2_test", metrics.get("R2_CV", 0))
    if r2 < 0.1:
        st.warning(
            f"⚠ **Ограничение модели ({context})**: R²(test) = {r2:+.3f}. "
            "Качество прогноза сопоставимо с предсказанием среднего значения "
            "обучающей выборки (baseline). Приложение носит демонстрационный "
            "характер и **не предназначено** для использования при проектировании "
            "реальных изделий без дополнительной валидации на расширенном датасете. "
            "Подробное обоснование приведено в разделах 2.3 и 2.4 пояснительной "
            "записки."
        )


# --------------------------------------------------------------------------- #
st.set_page_config(page_title="Прогноз свойств композитов", layout="wide")
st.title("Прогнозирование конечных свойств композиционных материалов")
st.caption(
    "Выпускная квалификационная работа по курсу «Data Science Pro», "
    "МГТУ им. Н. Э. Баумана, 2025"
)

art = load_artifacts()
metadata = art["metadata"]

tab1, tab2 = st.tabs(
    ["📊 Прогноз механических свойств",
     "🎛 Рекомендация соотношения матрица-наполнитель"]
)

# ---------------------------- Вкладка 1 ------------------------------------ #
with tab1:
    st.subheader("Ввод технологических параметров")
    st.caption(
        "Используются модели Ridge (для модуля упругости) и Random Forest "
        "(для прочности), выбранные по результатам 10-кратной кросс-валидации "
        "(подраздел 2.3 пояснительной записки)."
    )

    cols = st.columns(2)
    inputs_1 = {}
    for i, feat in enumerate(metadata["features_23"]):
        with cols[i % 2]:
            inputs_1[feat] = render_input(feat, "t1")

    left, right = st.columns([1, 1])
    reset = left.button("↺ Сбросить значения", key="reset_t1")
    if reset:
        for feat in metadata["features_23"]:
            st.session_state.pop(f"t1::{feat}", None)
        st.rerun()

    predict = right.button(
        "▶ Выполнить прогноз механических свойств", type="primary", key="predict_t1"
    )
    if predict:
        X_df = pd.DataFrame([[inputs_1[f] for f in metadata["features_23"]]],
                            columns=metadata["features_23"])
        X_scaled = art["scaler_23"].transform(X_df.values)
        pred_mod = float(art["ridge_mod"].predict(X_scaled)[0])
        pred_str = float(art["rf_str"].predict(X_scaled)[0])

        st.success("Прогноз выполнен.")
        m1, m2 = st.columns(2)
        m1.metric("Модуль упругости при растяжении, ГПа", f"{pred_mod:.2f}")
        m2.metric("Прочность при растяжении, МПа", f"{pred_str:.1f}")

        result_df = pd.DataFrame({
            "Параметр": ["Модуль упругости при растяжении, ГПа",
                         "Прочность при растяжении, МПа"],
            "Прогноз": [round(pred_mod, 2), round(pred_str, 1)],
        })
        st.download_button(
            "💾 Скачать результат (CSV)",
            data=result_df.to_csv(index=False).encode("utf-8"),
            file_name="prediction_mechanical.csv",
            mime="text/csv",
        )

        with st.expander("Метрики моделей (из metadata.json)"):
            st.json(metadata["metrics"])

        show_model_disclaimer(metadata["metrics"]["modulus"],
                              "модуль упругости")
        show_model_disclaimer(metadata["metrics"]["strength"], "прочность")


# ---------------------------- Вкладка 2 ------------------------------------ #
with tab2:
    st.subheader("Рекомендация соотношения матрица-наполнитель")
    st.caption(
        "Используется нейронная сеть MLP (PyTorch), обученная на 10 технологических "
        "параметрах без утечки данных — модуль упругости и прочность при растяжении "
        "из входов исключены (подраздел 2.4 пояснительной записки)."
    )

    cols = st.columns(2)
    inputs_2 = {}
    for i, feat in enumerate(metadata["features_24"]):
        with cols[i % 2]:
            inputs_2[feat] = render_input(feat, "t2")

    left, right = st.columns([1, 1])
    reset = left.button("↺ Сбросить значения", key="reset_t2")
    if reset:
        for feat in metadata["features_24"]:
            st.session_state.pop(f"t2::{feat}", None)
        st.rerun()

    predict = right.button(
        "▶ Получить рекомендацию", type="primary", key="predict_t2"
    )
    if predict:
        X = np.array([[inputs_2[f] for f in metadata["features_24"]]],
                     dtype=np.float32)
        X_scaled = art["scaler_mlp_X"].transform(X).astype(np.float32)
        with torch.no_grad():
            pred_s = art["mlp"](torch.tensor(X_scaled)).numpy().ravel()
        ratio = float(art["scaler_mlp_y"].inverse_transform(
            pred_s.reshape(-1, 1)).ravel()[0])

        st.success("Рекомендация сформирована.")
        st.metric("Рекомендуемое соотношение матрица-наполнитель", f"{ratio:.3f}")

        with st.expander("Метрики модели MLP (из metadata.json)"):
            st.json(metadata["metrics"]["ratio"])

        show_model_disclaimer(metadata["metrics"]["ratio"],
                              "соотношение матрица-наполнитель")


# ---------------------------- Футер --------------------------------------- #
st.divider()
st.caption(
    f"Версия: {metadata.get('version', '—')} | "
    f"random_state = {metadata.get('seed', '—')}"
)


## 5. Обучение моделей

Скрипт обучает три модели и сохраняет 7 артефактов в директорию `models/`. Занимает 1–2 минуты на стандартной ВМ Colab.


In [ ]:
!python train_models.py


## 6. Запуск Streamlit с публичным URL

### Вариант А — ngrok (рекомендуемый, требует бесплатной регистрации)

1. Зарегистрируйтесь на https://dashboard.ngrok.com/signup (бесплатно, без карты).
2. Скопируйте токен со страницы https://dashboard.ngrok.com/get-started/your-authtoken.
3. Выполните ячейку ниже, вставьте токен в появившееся поле ввода.


In [ ]:
import getpass, subprocess, time
from pyngrok import ngrok, conf

# Вставьте токен в поле ввода (символы не отображаются)
token = getpass.getpass('ngrok authtoken: ')
conf.get_default().auth_token = token

# Закрываем старые процессы и тоннели, если были
!pkill -f streamlit 2>/dev/null || true
ngrok.kill()

# Запускаем Streamlit в фоне
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.headless', 'true',
     '--server.port', '8501',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)  # даём Streamlit подняться

# Открываем публичный HTTPS-тоннель
public_url = ngrok.connect(addr='8501', proto='http', bind_tls=True)
print('➡ Streamlit доступен по адресу:', public_url)
print('  Откройте URL выше в новой вкладке браузера.')


### Вариант B — localtunnel (без регистрации)

Использовать, если не хочется регистрироваться в ngrok. localtunnel иногда запрашивает «tunnel password» при первом открытии URL — это публичный IP вашей ВМ Colab, он выводится в ячейке ниже.

**Запустите эту ячейку вместо предыдущей, а не вместе с ней.**


In [ ]:
!pkill -f streamlit 2>/dev/null || true
!npm install -g --silent localtunnel

import subprocess, time
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.headless', 'true',
     '--server.port', '8501',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)

print('Если localtunnel попросит «tunnel password» — используйте IP ниже:')
!curl -s https://ipv4.icanhazip.com
print()
!npx --yes localtunnel --port 8501


## 7. Остановка приложения

Когда закончите работу:


In [ ]:
!pkill -f streamlit 2>/dev/null || true
try:
    from pyngrok import ngrok
    ngrok.kill()
except Exception:
    pass
print('✓ Streamlit и ngrok остановлены')


## Ограничения и советы

- **Сессия Colab живёт ≈ 90 минут** без активности в браузере с ноутбуком и до 12 часов в сумме на бесплатном тарифе. При отключении все файлы в `data/` и `models/` пропадают.
- Бесплатный ngrok **меняет URL при каждом запуске** и допускает один одновременный тоннель.
- **Сохранение артефактов между сессиями** — подключите Google Drive и сохраняйте в него папку `models/`, либо запускайте `train_models.py` заново (занимает ≈1 мин).
- Если вкладка с приложением «зависла» — перезапустите ячейку запуска. Streamlit сам переподключается.
- **Для демонстрации комиссии** лучше предварительно развернуть приложение на Streamlit Community Cloud (бесплатно, постоянный URL), а Colab оставить как запасной вариант.
